# Exercise 4: Entity distribution shift measurement

*Sequence Labelling · version 0.1.0*

> Extract all PERSON entity spans from the KazNERD training set and a Turkish NER training set (for example the Starlang Turkish NER data or MilliyetNER). Normalise both sets (lemmatise or strip case suffixes, and transliterate to a common script with a documented mapping), then compute the Jaccard overlap of the name tokens and of their character trigrams. Which normalised strings are shared and which are not? Do not infer a name’s etymology from string overlap alone. Stratify Exercise 3’s PERSON mentions by whether their normalised name strings occur in the source training data: are unseen names the main source of cross-lingual PERSON errors?

**Setup.** `pip install "turkicnlp[hfst]==0.5.0" "stanza==1.11.0" sklearn-crfsuite transformers sentencepiece requests` (the compiled `hfst` package needs Python 3.11 or 3.12; elsewhere install `pyhfst`, which the first cell uses instead). Stanza is pinned to 1.11.0 because the worked solutions use its Kazakh KazNERD checkpoint; models already on disk are used from `STANZA_RESOURCES_DIR` (default `~/stanza_resources`), and missing ones are downloaded. The datasets are read from commit-pinned GitHub files (KazNERD, KyrgyzNER, the Starlang Turkish NER data, TRopBank, the Turkish WordNet and UD 2.17 IMST), downloaded once into `BOOK_DATA_DIR` (default `~/.cache/turkic-nlp-book`) and checked against SHA-256 hashes. Taggers receive the **gold words**, so tokenisation errors do not enter the scores.

**Normalisation.** Kazakh name tokens are reduced to the proper-noun stem of the Apertium analyser when it has one (removing case and possessive suffixes); Turkish name tokens need no stripping, because the Starlang data split suffixes off as separate tokens (*Şenliği 'ne*). Both sides are then transliterated to the Common Turkic Alphabet with TurkicNLP's `kaz Cyrl→CTS` and `tur Latn→CTS` mappings and lower-cased. The Turkish data is `nerdata.txt` of `StarlangSoftware/TurkishNamedEntityRecognition` (ENAMEX mark-up). The seen/unseen split uses the Kyrgyz zero-shot predictions of Exercise 3, so the Exercise 3 cell runs first; a Kyrgyz PERSON mention counts as *seen* if its final word, normalised in the same way (`kir Cyrl→CTS`), occurs among the KazNERD PERSON tokens.

In [ ]:
import collections, hashlib, os, random, re, statistics
from pathlib import Path
import requests

# Pinned public releases (commit-addressed raw files) and their SHA-256 hashes.
KAZNERD = "https://raw.githubusercontent.com/IS2AI/KazNERD/bd4333d0f5952b9fafb2ef2ac2fefa0ad3c0333f/KazNERD/IOB2_{}.txt"
KYRGYZNER = "https://raw.githubusercontent.com/Akyl-AI/KyrgyzNER/b76b9c9c9c4f5d681c25308cc0d8dd9f0bd6569c/data/{}"
SHA256 = {
    KAZNERD.format("train"): "e5a80f6d4bbff499a4bf7cbf96a169feb761411151fb45abebeaae06995ff647",
    KAZNERD.format("valid"): "29d584d1a0d9aca6d58b39eb8a93e154791d303ecff56ef93f12b3c889239bf1",
    KAZNERD.format("test"): "359a1f61f6a21280c6c47b4fde9a5adcd07411d62015aa6e32d940c51f163b7a",
    KYRGYZNER.format("KyrgyzNER_TRAIN.jsonl.txt"): "719a74a18cd99d6709c1e0592d0337aafade65aa97fe762e62f3f72a6b343a26",
    KYRGYZNER.format("KyrgyzNER_TEST.jsonl.gold.txt"): "57a35a92139497819a2792d3e9e00a41b387b029b7851be7a64f4968befaddf5",
}
CACHE = Path(os.environ.get("BOOK_DATA_DIR", "~/.cache/turkic-nlp-book")).expanduser()

def fetch(url, sha256=None):
    """Download a pinned file once, check its hash and return the local path."""
    sha256 = sha256 or SHA256[url]
    path = CACHE / hashlib.sha1(url.encode()).hexdigest()[:12] / url.rsplit("/", 1)[-1]
    if not path.exists():
        r = requests.get(url, timeout=300)
        r.raise_for_status()
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_bytes(r.content)
    if hashlib.sha256(path.read_bytes()).hexdigest() != sha256:
        raise ValueError(f"{url} differs from the pinned release")
    return path

def read_conll(path, tag_col=-1, comment="#"):
    """Sentences as (words, tags) from a column file with blank-line sentence breaks."""
    sents, words, tags = [], [], []
    for line in Path(path).read_text(encoding="utf-8").splitlines() + [""]:
        if not line.strip():
            if words:
                sents.append((words, tags))
            words, tags = [], []
        elif not line.startswith(comment + " "):
            cols = line.split("\t") if "\t" in line else line.split()
            words.append(cols[0]); tags.append(cols[tag_col].strip())
    return sents

def spans(tags):
    """Strict IOB2 spans {(start, end, type)}; an I- tag that does not continue a span starts none."""
    out, start, typ = set(), None, None
    for i, t in enumerate(list(tags) + ["O"]):
        if t.startswith("I-") and typ == t[2:]:
            continue
        if typ is not None:
            out.add((start, i, typ))
        start, typ = (i, t[2:]) if t.startswith("B-") else (None, None)
    return out

def bioes_to_iob2(tags):
    return ["B-" + t[2:] if t[:2] in ("S-", "B-") else "I-" + t[2:] if t[:2] in ("E-", "I-") else "O" for t in tags]

def prf(gold, pred):
    """Micro precision, recall and F1 (%) over lists of span sets."""
    g = sum(len(x) for x in gold); p = sum(len(x) for x in pred)
    c = sum(len(x & y) for x, y in zip(gold, pred))
    pr = c / p if p else 0.0; rc = c / g if g else 0.0
    return {"gold": g, "pred": p, "correct": c, "P": round(100 * pr, 2), "R": round(100 * rc, 2),
            "F1": round(100 * (2 * pr * rc / (pr + rc) if pr + rc else 0.0), 2)}

def stanza_ner(lang="kk"):
    """Stanza NER over gold words; models from STANZA_RESOURCES_DIR, downloaded only if missing."""
    import logging, stanza
    logging.getLogger("stanza").setLevel(logging.ERROR)
    model_dir = os.environ.get("STANZA_RESOURCES_DIR", str(Path("~/stanza_resources").expanduser()))
    have = (Path(model_dir) / lang / "ner").exists()
    return stanza.Pipeline(lang, processors="tokenize,ner", tokenize_pretokenized=True, model_dir=model_dir,
                           download_method=None if have else "reuse_resources", verbose=False)

def tag_words(nlp, sents, batch=2000):
    """IOB2 predictions for gold-word sentences (spaces inside a token are joined with '_')."""
    out = []
    for i in range(0, len(sents), batch):
        doc = nlp([[w.replace(" ", "_") for w in words] for words, _ in sents[i:i + batch]])
        for s, (words, _) in zip(doc.sentences, sents[i:i + batch]):
            assert len(s.tokens) == len(words)
            out.append(bioes_to_iob2([t.ner for t in s.tokens]))
    return out

In [ ]:
ky_train = read_conll(fetch(KYRGYZNER.format("KyrgyzNER_TRAIN.jsonl.txt")))
ky_test = read_conll(fetch(KYRGYZNER.format("KyrgyzNER_TEST.jsonl.gold.txt")))
types = lambda sents: collections.Counter(t[2:] for _, g in sents for t in g if t.startswith("B-"))
print(f"KyrgyzNER release: train {len(ky_train)} sentences, test {len(ky_test)} (no development split); "
      f"{len(set(types(ky_train)) | set(types(ky_test)))} entity types occur")

# The label mapping, declared before any score is computed. Types without a counterpart become O.
KAZ_MAP = {"PERSON": "PER", "GPE": "LOC", "LOCATION": "LOC", "ORGANISATION": "ORG"}
KIR_MAP = {"PERSON": "PER", "LOCATION": "LOC", "ORGANISATION": "ORG", "INSTITUTION": "ORG", "BUSINESS": "ORG"}

def mapped(tags, table):
    out = []
    for t in tags:
        m = table.get(t[2:]) if t != "O" else None
        out.append(f"{t[0]}-{m}" if m else "O")
    return out

gold = [spans(mapped(g, KIR_MAP)) for _, g in ky_test]
print("mapped test mentions:", collections.Counter(t for s in gold for *_, t in s))

# Zero-shot: the pretrained Stanza KazNERD model on Kyrgyz Cyrillic, with the same mapping.
zs = [spans(mapped(p, KAZ_MAP)) for p in tag_words(stanza_ner("kk"), ky_test)]
print("zero-shot Stanza KazNERD -> KyrgyzNER test:", prf(gold, zs))
for typ in ("PER", "LOC", "ORG"):
    print(f"   {typ}:", prf([{x for x in s if x[2] == typ} for s in gold], [{x for x in s if x[2] == typ} for s in zs]))

In [ ]:
try:
    import hfst
except ImportError:
    import sys, pyhfst
    sys.modules["hfst"] = pyhfst
import turkicnlp
from turkicnlp import Pipeline
from turkicnlp.scripts import Script
from turkicnlp.scripts.transliterator import Transliterator
STARLANG = ("https://raw.githubusercontent.com/StarlangSoftware/TurkishNamedEntityRecognition/"
            "9d0e105ed6d039b47ebbb1ab7c89e994d7aef104/nerdata.txt")
STARLANG_SHA = "5ad492aa0da4caeecd5cc9109ad0cc6fe1d3b17e32c2846d68e9ec397ccb679c"

def starlang_persons(path):
    """PERSON spans of the Starlang Turkish NER file (ENAMEX mark-up; suffixes are split off as tokens)."""
    text = Path(path).read_text(encoding="utf-8")
    return [m.split() for m in re.findall(r'<b_enamex TYPE="PERSON">(.*?)<e_enamex>', text)]

cts = {"kaz": Transliterator("kaz", Script("Cyrl"), Script("CTS")),
       "kir": Transliterator("kir", Script("Cyrl"), Script("CTS")),
       "tur": Transliterator("tur", Script("Latn"), Script("CTS"))}
pipes = {}
def stem(word, lang):
    """Proper-noun stem from the Apertium analyser when it has one (case and possessive removed)."""
    if lang not in pipes:
        turkicnlp.download(lang, processors=["morph"])
        pipes[lang] = next(p for p in Pipeline(lang, processors=["tokenize", "morph"], tokenize_backend="rule",
                                                morph_backend="apertium")._processors if hasattr(p, "_analyze"))
    for r in pipes[lang]._analyze(word):
        raw = r["raw"].split("+")[0]
        if "<np>" in raw and not raw.startswith("*"):
            return raw.split("<")[0]
    return word

def norm(word, lang):
    """Case-stripped (Kazakh, Kyrgyz) and CTS-transliterated, lower-cased name token."""
    w = stem(word, lang) if lang in ("kaz", "kir") else word
    return cts[lang].transliterate(w).lower()

kaz_train = read_conll(fetch(KAZNERD.format("train")))
kaz_names = collections.Counter(norm(w, "kaz") for ws, g in kaz_train
                                for s, e, t in spans(g) if t == "PERSON" for w in ws[s:e])
tur_names = collections.Counter(norm(w, "tur") for m in starlang_persons(fetch(STARLANG, STARLANG_SHA)) for w in m)
tri = lambda c: {w[i:i + 3] for w in c for i in range(max(1, len(w) - 2))}
jac = lambda a, b: len(a & b) / len(a | b)
print(f"PERSON name-token types: KazNERD train {len(kaz_names)}, Starlang Turkish {len(tur_names)}")
print(f"Jaccard overlap: name tokens {jac(set(kaz_names), set(tur_names)):.3f}, character trigrams {jac(tri(kaz_names), tri(tur_names)):.3f}")
shared = sorted(set(kaz_names) & set(tur_names), key=lambda w: -kaz_names[w])
print("most frequent shared tokens (KazNERD count / Turkish count):",
      ", ".join(f"{w} {kaz_names[w]}/{tur_names[w]}" for w in shared[:15]))
print("most frequent KazNERD tokens absent from the Turkish data:",
      ", ".join([f"{w} {c}" for w, c in kaz_names.most_common(60) if w not in tur_names][:15]))

# Seen vs unseen Kyrgyz PERSON mentions under zero-shot transfer from KazNERD
# (ky_test, gold and zs come from the Exercise 3 cell above).
seen = {True: [0, 0], False: [0, 0]}
for (ws, _), g, p in zip(ky_test, gold, zs):
    for s, e, t in g:
        if t == "PER":
            k = norm(ws[e - 1], "kir") in kaz_names        # the final word: the (sur)name
            seen[k][0] += 1; seen[k][1] += (s, e, t) in p
for k in (True, False):
    n, ok = seen[k]
    print(f"Kyrgyz PERSON mentions whose final word is {'seen' if k else 'unseen'} in KazNERD PERSON names: "
          f"n={n}, zero-shot recall {100 * ok / n:.1f}%")

## Worked solution

Output of `experiments/ch09_exercises/run_ch09_exercises.py`:

```
PERSON name-token types: KazNERD train 3392, Starlang Turkish 4082
Jaccard overlap: name tokens 0.016, character trigrams 0.315
most frequent shared tokens (KazNERD count / Turkish count): samat 127/3, sergey 91/3, ercan 71/7, mansur 69/17, aleksandr 45/1, aslan 37/18, murat 37/76, ayan 33/2, roman 31/1, sultan 27/10, eduard 23/1, timur 21/5, aynur 19/1, elena 17/2, dana 15/2
most frequent KazNERD tokens absent from the Turkish data: toqaev 736, qasım-comart 700, nazarbaev 667, nursultan 654, abay 362, gülcan 194, asqar 162, azamat 146, ruslan 131, riat 130, marqabaeva 128, candos 123, nurlan 121, i̇ndira 119, aqmaral 112
Kyrgyz PERSON mentions whose final word is seen in KazNERD PERSON names: n=236, zero-shot recall 51.3%
Kyrgyz PERSON mentions whose final word is unseen in KazNERD PERSON names: n=915, zero-shot recall 47.4%
```

**Overlap.** After normalisation there are 3,392 distinct PERSON tokens in KazNERD and 4,082 in the Turkish data, and only 1.6% of the union is shared (Jaccard 0.016). Character trigrams overlap much more (0.315): the names are built from similar pieces, but whole names rarely coincide.

**What is shared and what is not.** The shared tokens are mostly international and Russian given names that both corpora report (*sergey*, *aleksandr*, *roman*, *eduard*, *elena*) and given names of Arabic, Persian or Turkic origin (*murat*, *mansur*, *sultan*, *timur*, *aslan*, *ercan*, *aynur*). The most frequent Kazakh names missing from the Turkish data are the names of Kazakh politicians in the news (*toqaev*, *qasım-comart*, *nazarbaev*, *nursultan*) and Kazakh given names (*asqar*, *azamat*, *nurlan*, *aqmaral*). The mapping itself limits overlap: CTS keeps the Kazakh *q* (қ) that Turkish spells *k*, so *Asqar* and Turkish *Askar* stay different tokens; a looser mapping would raise the overlap and would also merge some names that are different.

**Seen vs. unseen in Kyrgyz.** Of the 1,151 Kyrgyz PERSON test mentions, 236 end in a name seen among KazNERD PERSON tokens; the Kazakh model recalls 51.3% of them, against 47.4% of the 915 unseen ones. Unseen names form the larger group and have lower recall. This split, based only on the final token, does not identify the main cause of errors or establish full-mention familiarity. Audit boundary mismatches and name membership separately; the second "Your turn" task investigates the boundary-policy difference.

## Your turn — extension

1. Replace CTS with a looser mapping (*q*→*k*, *ğ*→*g* …) and recompute the overlap. Which shared names are real, which are accidental?
2. Score the Kyrgyz PERSON mentions with a lenient criterion (any overlap with the gold span) and compare seen and unseen recall again.